# 07 Building a Local MCP Server

## 本地 MCP Server 不是演示道具，而是能力层第一次真正落地

到了这一章，问题已经不再是“我是否理解 MCP 的概念”，而是“我能不能把这些概念落成一个有结构的本地能力面”。这一步非常关键，因为很多关于 Agent、Tool Use、MCP 的讨论，一旦不进入实现，就很容易停留在听起来都对的抽象层。

做一个本地 MCP Server 的价值，不在于证明“server 能启动”，而在于验证前面几章里一系列判断是否真的成立：工具是否能被描述成稳定能力对象，资源是否能被组织成上下文入口，prompt 是否真的值得成为独立能力，Agent Runtime 未来到底要消费什么样的能力面。

所以，这一章不是教程式地带着读者跑通命令，而是把一个最小但像样的本地 MCP Server 当成系统设计对象来拆。

## 先给结论

如果把这一章的重点压缩成一句话，可以这样说：

> 一个本地 MCP Server 的意义，不是把几段本地代码挂出去，而是把本地世界重新整理成 Agent 可发现、可读取、可调用、可复用的一层统一能力面。

这个判断决定了本章的写法不会是“随便暴露几个函数”。真正应该关心的是：

- 为什么选这些能力，而不是别的能力
- 这些能力分别应该建模成 tool、resource 还是 prompt
- 它们的描述是否足够像协议对象，而不是应用内部实现细节
- 它们是否能支持后面的 Agent Runtime 和案例演示

换句话说，这一章关心的是能力建模，而不只是代码连通。

## 1. 为什么先做本地 MCP Server，而不是直接接远程系统

从展示效果看，远程服务当然更像真实生产环境；但从这套项目的目标看，本地 server 反而是更合适的第一站。

原因并不复杂：

- 本地环境更容易控制变量
- 可以直接围绕当前项目内容设计 resource 和 prompt
- 不需要把外部系统复杂性混进能力层设计讨论里
- 更适合清楚展示 Agent Runtime 究竟在消费什么

如果一开始就接远程平台，读者很容易把注意力转移到网络、权限、依赖和接入细节上，反而看不清 MCP Server 作为能力层原型的真正意义。本地 server 的价值，恰恰在于它把问题压回到结构本身。

## 2. 一个像样的本地 Server，目标不该是“功能多”，而该是“能力面清楚”

很容易犯的一个错误，是把本地 MCP Server 写成杂货铺：能暴露什么就暴露什么，工具越多越好，资源越多越显得完整。但这样做通常只会把结构弄脏。

对这套项目来说，一个像样的本地 server 更应该满足下面这些目标：

- 能清楚展示 tool、resource、prompt 的差异
- 能服务于后续的 Agent Runtime 案例
- 能反映“文义说明 + 执行接口”统一的设计思想
- 能让读者一眼看出能力面不是随手拼出来的

这意味着 server 不需要工具很多，但每个能力都应该有明确职责。与其暴露十个玩具函数，不如暴露三到五个真正构成系统骨架的能力对象。

## 3. 如何选择本地 Server 要暴露哪些能力

能力选择不是“想到什么挂什么”，而应该从后续任务链反推。

考虑这套项目后面会进入 Agent Runtime、HR 向案例、文档分析和需求拆解，那么本地 server 至少应该能支撑三类操作：

- 读取项目与案例相关文档
- 执行少量结构化操作
- 提供标准任务入口模板

一个比较稳的最小集合可以是这样：

- `tools`
  - `extract_key_requirements`：从文本中抽取结构化要求
  - `score_candidate_fit`：根据能力维度输出候选人匹配度
  - `summarize_section`：对长内容做压缩总结
- `resources`
  - 当前项目的 PRD
  - 架构说明或案例材料
  - 岗位 JD / 候选人样本这类演示输入
- `prompts`
  - `map_candidate_to_jd`
  - `analyze_project_brief`

这个集合并不追求通用性最大化，而是追求能力面的解释力和后续可演示性。

## 4. Tool 设计：不是暴露函数，而是暴露动作语义

在本地 server 里写工具最容易滑向一种程序员惯性：先看本地有什么函数能复用，再把它们挂出去。这种做法效率很高，但经常会把能力层做成实现细节清单，而不是协议对象集合。

更合理的顺序应该倒过来：先问后续 Agent 需要哪些动作语义，再决定底层怎么实现。

比如，一个名为 `parse_text_blob_v2` 的内部函数，也许实现很强，但它未必适合作为 MCP tool 名称；因为对 Agent 来说，这个名字几乎不表达动作意义。相反，如果工具叫 `extract_key_requirements`，再配一个明确说明“用于从职位描述、需求说明或项目文档中提取结构化要求”的描述，模型和 runtime 就都更容易理解何时应该用它。

这说明工具层设计的第一原则不是“复用现有函数名”，而是“暴露对任务有意义的动作语义”。

In [ ]:
tool_spec = {
    "name": "extract_key_requirements",
    "description": "从职位描述、项目需求或方案文档中提取结构化要求，用于后续匹配、评估和摘要任务。",
    "input_schema": {
        "type": "object",
        "properties": {
            "text": {"type": "string", "description": "原始文本内容"},
            "focus": {"type": "string", "description": "提取重点，例如 skills, risks, requirements"}
        },
        "required": ["text"]
    }
}

print(tool_spec)

## 5. Resource 设计：不是把文件路径暴露出去，而是把上下文入口设计出来

本地 server 一旦开始暴露 resource，很容易出现另一种偷懒方式：直接把一堆文件路径挂出去，仿佛这样就已经有了 resource 层。

这当然可以跑，但它仍然停留在“存储对象可达”层面，还没有进入“上下文入口被设计”层面。

对 Agent 来说，一个好 resource 至少应该做到几件事：

- 它有稳定标识，而不是依赖某个宿主的临时路径感知
- 它的描述能说明内容是什么、适合什么任务使用
- 它的组织方式能让 runtime 明白何时读它，而不是把所有文件一股脑放成目录索引

也就是说，resource 的本质不是“把文件读出来”，而是把文件、文档、知识片段重新包装成任务上下文入口。路径只是实现，入口才是设计。

In [ ]:
resource_spec = {
    "uri": "project://prd/main",
    "name": "Current Project PRD",
    "description": "当前项目的产品设计文档，用于解释整体目标、章节设计和交付范围。",
}

print(resource_spec)

## 6. Prompt 设计：不是给模型一段话，而是给 Agent 一个标准任务入口

本地 server 里如果要暴露 prompt，最忌讳的一种做法是把一段随手写的长提示词直接挂出去，既没有任务边界，也没有输入约束，只是把“模板”当作一个更长的字符串。

更合理的 prompt 设计应当体现三个层面：

- 它服务于什么任务类型
- 它需要哪些输入变量
- 它会把模型导入怎样的分析结构

例如，`map_candidate_to_jd` 这个 prompt 就不应该只是“请分析候选人与岗位匹配度”，而应该明确：

- 输入包括 JD 文本、候选人经历、关注维度
- 输出应该按能力维度、证据、风险、追问点来组织
- 如果信息不足，应优先指出空缺而不是硬凑结论

当 prompt 被这样设计后，它才真正像一个可复用的任务入口，而不是调用方私下藏着的一段好用文案。

## 7. 本地 Server 的边界：不要把 Agent Runtime 写进 Server

一个很容易出现的结构错误，是因为 MCP server 和 Agent runtime 最终都会出现在同一个项目里，于是开发时不自觉地把两者混起来：server 不只暴露能力，还顺手开始做任务判断、流程编排、状态推进。

这会直接破坏层次。

本地 server 的职责应该保持克制：

- 它负责暴露能力对象
- 它负责定义这些对象的输入输出契约
- 它负责返回能力执行或读取结果

而它不应该负责：

- 决定任务下一步做什么
- 管理跨步骤状态
- 判断整体目标是否完成

这些事情都应该留给后面的 Agent Runtime。否则，能力层和任务层一混，整个系统很快会重新退回私有胶水结构。

## 8. 一个最小但像样的本地 Server，应该长什么样

就这套项目而言，一个最小但像样的 server 可以遵循如下结构：

- 暴露 2 到 3 个工具，分别覆盖抽取、评分、总结这类动作
- 暴露 2 到 4 个资源，分别覆盖 PRD、案例材料、岗位或候选人样本
- 暴露 1 到 2 个 prompt 模板，分别覆盖岗位映射和项目分析两类场景

它不需要复杂，但必须体现出三种能力对象的边界清晰：

- tool 负责动作
- resource 负责材料
- prompt 负责入口

一旦这三层边界被立住，后面的 Agent Runtime 就可以开始像消费真正能力面那样来消费它，而不是继续面对一堆局部工具和字符串常量。

In [ ]:
local_mcp_surface = {
    "tools": [
        "extract_key_requirements",
        "score_candidate_fit",
        "summarize_section",
    ],
    "resources": [
        "project://prd/main",
        "project://case/jd_sample",
        "project://case/candidate_profile",
    ],
    "prompts": [
        "map_candidate_to_jd",
        "analyze_project_brief",
    ],
}

for key, value in local_mcp_surface.items():
    print(f"{key}: {value}")

## 9. 输入校验和错误返回，应该在本地 Server 层就被严肃对待

因为这是本地演示项目，最容易被忽略的一点就是错误处理。很多人会默认“反正都是自己写的输入，先不管异常路径”。但如果目标是做一套能体现工程判断力的作品，这种偷懒会很明显。

对一个像样的本地 MCP Server 来说，至少要考虑：

- tool 输入不完整时返回什么
- resource 不存在时如何说明
- prompt 输入变量缺失时如何报错
- 结果是否应该带上足够让 runtime 继续决策的错误信息

这不只是为了健壮性，也是为了后面的 Agent Runtime 能区分“失败了”与“该换路径了”。如果 server 只会把一切失败都压成模糊字符串，runtime 很难做出像样的恢复决策。

## 10. 命名一致性，是能力层质量的一部分

在本地 server 这种小系统里，最容易被人忽视、但最能暴露质量感的，是命名。

如果 tools 用动词短语、resources 用 URI 风格、prompts 用任务入口风格，整个能力面会显得像一个被设计过的系统；如果命名风格混乱，比如有的像内部函数名、有的像文件名、有的像聊天短句，读者很快就会感觉这是临时拼装的。

命名并不是审美问题，而是能力面是否可被稳定理解的问题。对模型、对 runtime、对后来维护的人来说，命名一致就是低成本的结构提示。

## 11. 怎样避免把本地 Server 做成玩具

“玩具感”通常不是因为功能少，而是因为能力对象和后续任务脱节。比如暴露一个天气工具、一个随机数工具、一个 hello world 资源，虽然也能展示协议形式，但和整套项目要解决的问题没有任何内在联系，读者会马上感觉这只是为了演示接口存在。

避免玩具感的方法很直接：让能力对象直接服务后续主线。

对这套项目来说，能力对象应该明显围绕：

- 文档读取
- 要求抽取
- 匹配分析
- 项目/岗位案例组织

这样到后面做 Agent Runtime 和 HR 向案例时，读者能一眼看出：这个 server 不是摆设，而是整个能力层的基础设施。

## 12. 本地 Server 与后续 Agent Runtime 的衔接关系

这一章真正要为后面铺的路，不是“server 已经有了”，而是“runtime 终于有东西可以消费了”。

后面的 Agent Runtime 一旦开始运行，至少会做几类事情：

- 先枚举可用能力
- 判断当前问题是先读 resource、先用 prompt、还是先调 tool
- 消费返回结果，并将其写回任务状态
- 再决定下一步动作

如果本地 server 做得足够清楚，runtime 这一层就能真正建立在能力面上；如果本地 server 只是若干临时接口，runtime 很快又会被迫写回一堆能力特判。前者是体系，后者是堆功能。

## 13. 从作品角度看，本地 Server 这一章到底证明了什么

从招聘展示视角，这一章的价值不只是“你会写一个 MCP server”。真正被展示出来的是另外几种能力：

- 你知道如何从任务主线反推能力建模
- 你知道能力层和任务层应该如何分开
- 你知道为什么 resource 和 prompt 不该被忽略
- 你知道一个小型原型也应该有清晰的契约、命名和错误处理

这些判断力比单纯把 server 跑起来更重要。因为会连接口的人很多，能把接口放进一个结构里的人少得多。

## 14. 本章结论

这一章最值得保留的判断有这些：

- 本地 MCP Server 的价值在于把本地世界组织成统一能力面，而不是证明 server 能启动。
- 能力选择应从后续任务链反推，而不是从手头函数清单正推。
- tool、resource、prompt 的设计重点分别是动作语义、上下文入口和任务入口。
- Server 负责暴露能力，不负责接管 Agent Runtime 的任务逻辑。
- 一个不像玩具的本地 server，关键不在能力多，而在能力对象与主线任务强相关。

下一章会把能力层真正接到本地模型上，也就是进入 `Ollama + gpt-oss:120b` 的本地接入与能力验证，开始让前面的理论结构进入实际运行链。